In [1]:
!pip install -q -U accelerate transformers pandas numpy torch scikit-learn matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 87.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 82.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 10.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 9.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 29.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Phase 2: Technique Prediction Pipeline
Once the Tactic (the "Why") is identified, the pipeline predicts the specific Technique (the "How") to map the text to the MITRE ATT&CK® framework.How it Works:Contextual Filtering: The model uses the predicted Tactic to narrow down the search space to a specific subset of techniques.Transformer-Based Classification: A fine-tuned model (e.g., SecureBERT) analyzes technical indicators like registry keys or API calls to assign a Technique ID.

Confidence Thresholding: Labels are only assigned if the model's probability score exceeds a set threshold (e.g., P > 0.85).Output:

Tactic (TAXXXX) -> Technique (TXXXX)

In [1]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/jamieteh/dataset-for-second-testing/complete_clean_training_dataset_part.csv")


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Starting Data Prep for Context-Aware Technique Classifier...")

# ==========================================
# FILTER RARE CLASSES (To prevent split errors)
# ==========================================
label_counts = df['labels'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df = df[df['labels'].isin(valid_labels)]

# ==========================================
# MAPPING 1: ENRICH THE INPUT (Context + Sentence)
# ==========================================
# This creates a new column combining both pieces of information
df['model_input'] = df['tactic'] + " " + df['sentence']

# ==========================================
# MAPPING 2: ENCODE THE TARGET LABELS
# ==========================================
technique_encoder = LabelEncoder()

# Translate strings like 'T1027' into numbers like 42
df['encoded_labels'] = technique_encoder.fit_transform(df['labels'])

# Build the mandatory PyTorch mappings
id2label = {i: label for i, label in enumerate(technique_encoder.classes_)}
label2id = {label: i for i, label in enumerate(technique_encoder.classes_)}

print(f"Total unique techniques mapped: {len(technique_encoder.classes_)}")

# ==========================================
# CREATE THE TRAIN/TEST SPLIT
# ==========================================
X = df['model_input'].tolist()
y = df['encoded_labels'].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n--- Mapping Verification ---")
print(f"Input Text: {X_train[0]}")
print(f"Target Label (Encoded): {y_train[0]}")
print(f"Target Label (Decoded): {id2label[y_train[0]]}")

Starting Data Prep for Context-Aware Technique Classifier...
Total unique techniques mapped: 255

--- Mapping Verification ---
Input Text: Execution The summit is the latest in a line of signs of diplomatic outreach from North Korea, following the Panmunjom Declaration for Peace, Prosperity and Unification of the Korean Peninsula between South Korea and North Korea on April 27, 2018
Target Label (Encoded): 58
Target Label (Decoded): T1059


In [3]:
import os
# --- THE FIX: Hide the second GPU from PyTorch to prevent the ModernBERT splitting bug ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
import pandas as pd
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

print("Initializing Context-Aware Technique Classifier...")

# 1. Initialize the SecureBERT model for Stage 2
# We use len(technique_encoder.classes_) which maps exactly to your ~293 techniques
stage2_model = AutoModelForSequenceClassification.from_pretrained(
    "cisco-ai/SecureBERT2.0-base", 
    num_labels=len(technique_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

tokenizer = AutoTokenizer.from_pretrained("cisco-ai/SecureBERT2.0-base")

# 2. Setup the PyTorch Datasets using the NEW Enriched Encodings
# Notice we are now tokenizing X_train and X_val (which contain Tactic + Sentence)
train_encodings_s2 = tokenizer(X_train, truncation=True, max_length=256)
val_encodings_s2 = tokenizer(X_val, truncation=True, max_length=256)

class TTPDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# We pass y_train and y_val (the encoded technique IDs)
train_dataset_s2 = TTPDataset(train_encodings_s2, y_train)
val_dataset_s2 = TTPDataset(val_encodings_s2, y_val)

# 3. Stage 2 Training Arguments 
training_args_s2 = TrainingArguments(
    output_dir='./results_stage2',
    num_train_epochs=4,              
    learning_rate=2e-5,               
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_dir='./logs_stage2',
    logging_steps=20,
    
    # --- KAGGLE DISK SPACE PROTECTION ---
    eval_strategy="epoch",
    save_strategy="no",               # Do not save massive checkpoints mid-training
    load_best_model_at_end=False,     # Must be false if we aren't saving intermediate checkpoints
    # ------------------------------------
    
    fp16=True,                        
    torch_compile=False,              
    label_smoothing_factor=0.1,       
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    # Weighted F1 is crucial for TTP because some tactics are rarer than others
    f1 = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}


# 4. Initialize the Trainer
# 4. Initialize the Trainer
trainer_s2 = Trainer(
    model=stage2_model,
    args=training_args_s2,
    train_dataset=train_dataset_s2,
    eval_dataset=val_dataset_s2,
    processing_class=tokenizer,  # <--- THE FIX!
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics=compute_metrics, 
)


# 5. START STAGE 2 TRAINING
print("🚀 Starting Stage 2 Training (Context + Sentence -> Specific Technique)...")
trainer_s2.train()

# 6. Save the final Technique Model
stage2_save_path = "/kaggle/working/model/"
stage2_model.save_pretrained(stage2_save_path)
tokenizer.save_pretrained(stage2_save_path) 
print(f"✅ Stage 2 Training Complete! Model safely save to {stage2_save_path}")

Initializing Context-Aware Technique Classifier...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: cisco-ai/SecureBERT2.0-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

🚀 Starting Stage 2 Training (Context + Sentence -> Specific Technique)...


W0422 19:39:10.267000 55 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.235903,2.111256,0.668756,0.625243
2,1.569019,1.817946,0.737223,0.713358
3,1.307036,1.709037,0.772420,0.753140
4,1.090129,1.707264,0.772903,0.754873


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Stage 2 Training Complete! Model safely save to /kaggle/working/model/


In [5]:
# This is a Jupyter Magic Command. It intercepts the hardware BEFORE PyTorch loads.
%env CUDA_VISIBLE_DEVICES=0

import torch
print(f"GPUs visible to PyTorch: {torch.cuda.device_count()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

env: CUDA_VISIBLE_DEVICES=0
GPUs visible to PyTorch: 1
GPU Name: Tesla T4


In [6]:
import numpy as np
from sklearn.metrics import classification_report
from collections import Counter

print("\n📊 Running Formal Evaluation on Validation Data...")

# 1. Ask the Trainer to predict on the validation dataset
evaluation_results = trainer_s2.predict(val_dataset_s2)

# 2. Extract the raw logits (math scores) and convert them to predicted Class IDs
predicted_logits = evaluation_results.predictions
predicted_ids = np.argmax(predicted_logits, axis=-1)

# 3. Extract the actual true answers
true_labels = evaluation_results.label_ids

# ==========================================
# NEW STEP: Extract Top 10 Techniques by Support
# ==========================================
# Count how many times each true label appears in the validation set
label_counts = Counter(true_labels)

# Get the 10 most common label IDs and their counts
top_10_items = label_counts.most_common(90)

# Extract just the IDs into a list
top_10_ids = [item[0] for item in top_10_items]

# Translate those IDs back into readable Technique Names (e.g., "T1059")
top_10_names = [technique_encoder.classes_[i] for i in top_10_ids]

# 4. Generate the targeted academic Classification Report
# We explicitly pass ONLY our top 10 IDs and Names to the report generator
report = classification_report(
    true_labels, 
    predicted_ids, 
    labels=top_10_ids, 
    target_names=top_10_names, 
    zero_division=0 
)

print("\n" + "="*60)
print(" STAGE 2: TOP 10 TECHNIQUES CLASSIFICATION REPORT ")
print("="*60)
print(report)

# 5. Save the report to a text file for your FYP Dissertation
report_path = "/kaggle/working/stage2_top10_evaluation_report.txt"
with open(report_path, "w") as f:
    f.write("STAGE 2: TOP 10 TECHNIQUES CLASSIFICATION REPORT\n")
    f.write("="*60 + "\n")
    f.write(report)

print(f"\n✅ Curated Top 10 academic report saved to: {report_path}")


📊 Running Formal Evaluation on Validation Data...



 STAGE 2: TOP 10 TECHNIQUES CLASSIFICATION REPORT 
              precision    recall  f1-score   support

       T1105       0.89      0.92      0.91       106
       T1027       0.85      0.95      0.90        92
       T1082       0.85      0.89      0.87        82
       T1083       0.81      0.92      0.86        72
       T1016       0.93      0.91      0.92        58
   T1070.004       0.81      0.86      0.83        58
   T1071.001       0.88      0.91      0.90        58
   T1547.001       0.84      0.95      0.89        55
   T1059.003       0.72      0.81      0.76        53
       T1033       0.97      0.88      0.92        40
   T1059.001       0.89      0.85      0.87        39
   T1053.005       0.86      0.97      0.91        32
       T1059       0.77      0.75      0.76        32
       T1005       0.79      0.84      0.82        32
   T1036.005       0.58      0.52      0.55        29
   T1204.002       0.63      0.83      0.72        29
       T1071       0.63      